In [1]:
import pandas as pd
import numpy as np
import re

from scipy.sparse import hstack

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import f1_score, confusion_matrix

In [2]:
df = pd.read_csv("/kaggle/input/competitions/comment-category-prediction-challenge/train.csv")

# fill NaN comment
df["comment"] = df["comment"].fillna("")

X = df.drop("label", axis=1)
y = df["label"]

In [3]:
# Part A (stratified)
x_train_A, x_val_A, y_train_A, y_val_A = train_test_split(
    X, y,
    test_size=0.4,
    random_state=2306,
    stratify=y
)

train_counts_A = y_train_A.value_counts().sort_index().to_numpy()
val_counts_A = y_val_A.value_counts().sort_index().to_numpy()


# Part B (non-stratified)
x_train_B, x_val_B, y_train_B, y_val_B = train_test_split(
    X, y,
    test_size=0.4,
    random_state=2306,
    stratify=None
)

train_counts_B = y_train_B.value_counts().sort_index().to_numpy()
val_counts_B = y_val_B.value_counts().sort_index().to_numpy()


# Part C
val_prop_A = val_counts_A / val_counts_A.sum()
val_prop_B = val_counts_B / val_counts_B.sum()

max_diff = np.max(np.abs(val_prop_A - val_prop_B))

print("Q1 Answer:", round(max_diff, 6))


Q1 Answer: 0.000606


In [4]:
# Use stratified split
x_train, x_test, y_train, y_test = x_train_A.copy(), x_val_A.copy(), y_train_A.copy(), y_val_A.copy()

# Step 1: Drop created_date
x_train = x_train.drop(columns=["created_date"])
x_test = x_test.drop(columns=["created_date"])


# Step 2: Separate text column
text_x_train = x_train["comment"]
text_x_test = x_test["comment"]

x_train = x_train.drop(columns=["comment"])
x_test = x_test.drop(columns=["comment"])


# Step 3: ColumnTransformer

categorical_cols = ["race", "religion", "gender", "disability"]

numerical_cols = [
    "post_id",
    "emoticon_1",
    "emoticon_2",
    "emoticon_3",
    "upvote",
    "downvote",
    "if_1",
    "if_2"
]

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_pipeline, categorical_cols),
        ("num", numeric_pipeline, numerical_cols)
    ],
    remainder="passthrough"
)

x_train_tabular = preprocessor.fit_transform(x_train)
x_test_tabular = preprocessor.transform(x_test)


# -----------------------------------------------------------
# Step 4: Text Cleaning
# -----------------------------------------------------------

def normalize_text(text):
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\s+', '  ', text).strip()
    return text

text_x_train_norm = text_x_train.apply(normalize_text)
text_x_test_norm = text_x_test.apply(normalize_text)

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)
tf_idf_train = tfidf.fit_transform(text_x_train_norm)
tf_idf_test = tfidf.transform(text_x_test_norm)


# -----------------------------------------------------------
# Step 6: Combine Features
# -----------------------------------------------------------

X_train_final = hstack([x_train_tabular, tf_idf_train])
X_test_final = hstack([x_test_tabular, tf_idf_test])

print("Q2 Answer:", round(X_train_final.sum(), 3))

Q2 Answer: 904262.933


In [5]:
svd = TruncatedSVD(n_components=300, random_state=2306)

X_train_reduced = svd.fit_transform(X_train_final)
X_test_reduced = svd.transform(X_test_final)

rf = RandomForestClassifier(random_state=2306)

param_dist = {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, 15]
}

randomized_search = RandomizedSearchCV(
    rf,
    param_dist,
    n_iter=5,
    cv=3,
    random_state=2306,
    n_jobs=-1
)

randomized_search.fit(X_train_reduced, y_train)

print("Q3 Answer:", randomized_search.best_params_["n_estimators"])

Q3 Answer: 200


In [6]:
ada = AdaBoostClassifier(
    n_estimators=50,
    random_state=2306
)

ada.fit(X_train_reduced, y_train)

variance = np.var(ada.estimator_errors_)

print("Q4 Answer:", round(variance, 4))

Q4 Answer: 0.0058


In [7]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=2306
)

rf_model.fit(X_train_reduced, y_train)

importances = rf_model.feature_importances_

max_index = np.argmax(importances)

print("Q5 Answer:", int(max_index))

Q5 Answer: 4


In [8]:
N = X_train_reduced.shape[1]

weights = (
    N * 128 +
    128 * 64 +
    64 * 32 +
    32 * 4
)

print("Q6 Answer:", int(weights))

Q6 Answer: 48768


In [9]:
mlp_q7 = MLPClassifier(
    hidden_layer_sizes=(128,64,32),
    activation="relu",
    solver="adam",
    max_iter=5,
    batch_size=32,
    random_state=2306
)

mlp_q7.fit(X_train_reduced, y_train)

print("Q7 Answer:", round(mlp_q7.loss_, 4))

Q7 Answer: 0.3271


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5) reached and the optimization hasn't converged yet.
  warnings.warn(


In [10]:
mlp_A = MLPClassifier(
    hidden_layer_sizes=(100,),
    max_iter=5,
    alpha=0.0001,
    random_state=2306
)

mlp_B = MLPClassifier(
    hidden_layer_sizes=(100,),
    max_iter=5,
    alpha=1.0,
    random_state=2306
)

mlp_A.fit(X_train_reduced, y_train)
mlp_B.fit(X_train_reduced, y_train)

pred_A = mlp_A.predict(X_train_reduced)
pred_B = mlp_B.predict(X_train_reduced)

f1_A = f1_score(y_train, pred_A, average="macro")
f1_B = f1_score(y_train, pred_B, average="macro")

diff = abs(f1_A - f1_B)

print("Q8 Answer:", round(diff, 4))

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5) reached and the optimization hasn't converged yet.
  warnings.warn(


Q8 Answer: 0.1327


In [11]:
mlp_val = mlp_q7

pred_val = mlp_val.predict(X_test_reduced)

cm = confusion_matrix(y_test, pred_val)

misclassified = cm.sum() - np.trace(cm)

ratio = misclassified / cm.sum()

print("Q9 Answer:", round(ratio, 4))

Q9 Answer: 0.1126
